# Cooking Assistant — Gemma 3 1B IT → MediaPipe `.task`

**Model:** `google/gemma-3-1b-it` (HuggingFace) → `DYNAMIC_INT8` TFLite → CPU `.task`

## Recommended: Try the fast path first

> **Cell 0** — downloads a pre-converted `dynamic_int8` `.task` directly from `litert-community` (~1 GB).  
> If it works: add to Xcode and you're done. No conversion pipeline needed.

## Full conversion workflow (if Cell 0 fails)

| Session | Cells | Purpose |
|---------|-------|---------|
| **Main** | 1 → 2 → 3 → 4 → 5 | Setup + download weights + inference test |
| **litert-torch** *(fresh runtime after Factory reset)* | 9-alt Step 1 → Step 2 | TFLite conversion |
| **Main** *(back)* | 9b → 10d → 10b → 11 | Package `.task` + download |

> **Why two sessions?** litert-torch requires a different TF/JAX stack than MediaPipe.  
> litert-torch cells must run in a completely fresh Colab runtime (after Factory reset).

---
## ⚡ Fast Path — Skip conversion entirely

`litert-community` publishes pre-converted Gemma 3 1B IT `.task` files directly on HuggingFace.  
**Run Cell 0 first.** If it succeeds, download the `.task` and skip everything else.

| File | Quantization | Context | Size |
|------|-------------|---------|------|
| `Gemma3-1B-IT_multi-prefill-seq_q8_ekv1280.task` | dynamic_int8 ✅ | 1280 tokens | ~1 GB |

Requirements: HuggingFace token with `google/gemma-3-1b-it` license accepted.

In [ ]:
## Cell 0 — Fast path: download pre-converted Gemma 3 1B IT (dynamic_int8, ~1 GB)
# If this cell succeeds: download cooking_assistant_gemma3.task → add to Xcode → done.
# If you get a 401/403: accept the license at https://huggingface.co/google/gemma-3-1b-it first.

import os
from google.colab import drive, files

drive.mount('/content/drive')

# ── HF token ─────────────────────────────────────────────────────────────────
HF_TOKEN_FILE = '/content/drive/MyDrive/hf_token.txt'
if os.path.exists(HF_TOKEN_FILE):
    with open(HF_TOKEN_FILE) as _f:
        HF_TOKEN = _f.read().strip()
    print("✓ HF token loaded from Drive")
else:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        print("✓ HF token loaded from Colab secrets")
    except Exception:
        HF_TOKEN = input("Paste your HuggingFace token (hf_...): ").strip()

# ── Download pre-converted dynamic_int8 .task (~1 GB) ────────────────────────
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download

REMOTE_FILE = 'Gemma3-1B-IT_multi-prefill-seq_q8_ekv1280.task'
FINAL_TASK  = '/content/cooking_assistant_gemma3.task'
DRIVE_DEST  = '/content/drive/MyDrive/cooking_assistant_gemma3.task'

print(f"Downloading {REMOTE_FILE} from litert-community (~1 GB)...")
downloaded = hf_hub_download(
    repo_id   = 'litert-community/Gemma3-1B-IT',
    filename  = REMOTE_FILE,
    token     = HF_TOKEN,
    local_dir = '/content',
)
os.rename(downloaded, FINAL_TASK)

task_mb = os.path.getsize(FINAL_TASK) / 1e6
print(f"✓ {task_mb:.0f} MB — saved as {FINAL_TASK}")

# ── Validate .task structure ──────────────────────────────────────────────────
import zipfile
with zipfile.ZipFile(FINAL_TASK, 'r') as z:
    entries = z.namelist()
    print("Contents:")
    for e in entries:
        print(f"  {e:50s} {z.getinfo(e).file_size/1e6:.1f} MB")
    has_tflite    = any('PREFILL_DECODE' in e or e.endswith('.tflite') for e in entries)
    has_tokenizer = any('TOKENIZER' in e.upper() or e.endswith('.model') or e.endswith('.json') for e in entries)
    print()
    print(f"  TF_LITE_PREFILL_DECODE : {'✓' if has_tflite    else '✗ MISSING'}")
    print(f"  TOKENIZER              : {'✓' if has_tokenizer else '✗ MISSING'}")

# ── Save to Drive + download ──────────────────────────────────────────────────
import shutil
shutil.copy(FINAL_TASK, DRIVE_DEST)
print(f"\n✓ Backed up to Drive: {DRIVE_DEST}")

print("\nDownloading to your machine...")
files.download(FINAL_TASK)

print("\n" + "="*55)
print("DONE — no conversion needed!")
print()
print("iOS integration:")
print("  1. Drag cooking_assistant_gemma3.task into Xcode")
print("  2. AppConfig.modelFileName already = 'cooking_assistant_gemma3'")
print("  3. Delete app from iPhone to clear Documents/ cache")
print("  4. Build & run")
print("="*55)


## Cell 1 — Install dependencies (main session)

Run once. Auto-restarts when done.

In [ ]:
!pip uninstall -y tensorflow tensorflow-gpu keras keras-nightly -q 2>/dev/null; echo done
!pip install -q \
    "mediapipe==0.10.21" \
    "tensorflow>=2.17.0" \
    "keras>=3.0.0" \
    "keras-hub" \
    "huggingface_hub" \
    "numpy<2"
print("\n" + "="*50)
print("Cell 1 done — restarting runtime...")
print("After restart run Cell 2 → 3 → 4 → 5")
print("="*50)
import os; os.kill(os.getpid(), 9)


## Cell 2 — Set JAX env vars

In [ ]:
import os
# Prevent JAX from pre-allocating all GPU memory
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.5'
print("✓ JAX env vars set")


## Cell 3 — Verify environment

In [ ]:
import tensorflow as tf
import mediapipe
import torch
import keras_hub
print(f"tensorflow  {tf.__version__}")
print(f"mediapipe   {mediapipe.__version__}")
print(f"torch       {torch.__version__}")
print(f"keras_hub   {keras_hub.__version__}")
print("✓ Environment OK")


## Cell 4 — Mount Drive + download Gemma 3 1B IT weights

Requires a HuggingFace token with access to `google/gemma-3-1b-it`:
1. Accept license at https://huggingface.co/google/gemma-3-1b-it
2. Create token at https://huggingface.co/settings/tokens (read access)
3. Save as `hf_token.txt` in your Drive root (just the token, no quotes)  
   *OR* add as Colab secret named `HF_TOKEN`


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── HF token ─────────────────────────────────────────────────────────────────
HF_TOKEN_FILE = '/content/drive/MyDrive/hf_token.txt'
if os.path.exists(HF_TOKEN_FILE):
    with open(HF_TOKEN_FILE) as _f:
        HF_TOKEN = _f.read().strip()
    print("✓ HF token loaded from Drive")
else:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        print("✓ HF token loaded from Colab secrets")
    except Exception:
        HF_TOKEN = input("Paste your HuggingFace token (hf_...): ").strip()

os.environ['HF_TOKEN'] = HF_TOKEN

# ── Download weights ──────────────────────────────────────────────────────────
from huggingface_hub import snapshot_download
print("Downloading google/gemma-3-1b-it (~2 GB, cached on repeat runs)...")
KERAS_DIR = snapshot_download(
    repo_id='google/gemma-3-1b-it',
    token=HF_TOKEN,
    ignore_patterns=['*.gguf', 'flax_model*', 'tf_model*', 'rust_model*'],
)
print(f"✓ Weights at: {KERAS_DIR}")
print(f"  Files: {[f for f in os.listdir(KERAS_DIR) if not f.startswith('.')]}")


## Cell 5 — Inference gate

Quick test to confirm the model weights are valid before conversion.  
Uses HuggingFace transformers (no keras_hub needed).  
**Skip to Cell 9-alt if you've already verified the model.**


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Loading Gemma 3 1B IT for inference test...")
tokenizer = AutoTokenizer.from_pretrained(KERAS_DIR, token=HF_TOKEN)
hf_model  = AutoModelForCausalLM.from_pretrained(
    KERAS_DIR,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=HF_TOKEN,
)
print("✓ Model loaded")

# Gemma 3 uses same chat template as Gemma 2
TEST = (
    '<start_of_turn>user\n'
    'Give me a simple pasta recipe\n'
    '<end_of_turn>\n'
    '<start_of_turn>model\n'
)
inputs = tokenizer(TEST, return_tensors='pt').to(hf_model.device)
with torch.no_grad():
    out = hf_model.generate(**inputs, max_new_tokens=200, do_sample=False)
result = tokenizer.decode(out[0], skip_special_tokens=True)
print("\n--- Model output ---")
print(result)
print("---")

del hf_model
import gc; gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n" + "="*50)
print("READ BEFORE CONTINUING:")
print("  ✓ Coherent recipe → proceed to Cell 9-alt")
print("  ✗ Garbled/empty  → STOP. Check KERAS_DIR.")
print("="*50)


---
## ⚡ litert-torch session — TFLite conversion

**Before running Step 1:**
1. `Runtime → Factory reset runtime` (completely clears all installed packages)
2. Run **Step 1** — it installs the litert-torch stack and auto-restarts
3. Run **Step 2** — converts Gemma 3 to TFLite

**After Step 2 completes**, switch back to the main session and run Cell 9b → 10d → 10b → 11.


## Cell 9-alt Step 1 — Install litert-torch stack

> **Requires Factory reset runtime first.** Auto-restarts when done.

In [ ]:
# ── Remove system TF/JAX that would conflict with litert-torch's TF ───────────
!pip uninstall -y tensorflow tensorflow-cpu tensorflow-gpu tensorflow-text \
    keras keras-nightly jax jaxlib jax-cuda12-plugin flax -q 2>/dev/null
!echo "Uninstall done"

# ── Install litert-torch stack ────────────────────────────────────────────────
# No TF version pin — litert-torch resolves the compatible version automatically.
# torchao 0.13.0 is required for pt2e quantization (added in 0.11.0).
!pip install -q \
    "torch==2.8.0" \
    "torchvision==0.23.0" \
    "torchaudio==2.8.0" \
    "torchao==0.13.0" \
    "litert-torch==0.8.0" \
    "huggingface_hub"
!echo "Install done"

# ── Do NOT import here — restart first so new .so files load cleanly ──────────
print("\n" + "="*55)
print("Step 1 complete — restarting runtime now...")
print("After restart: run Step 2 ONLY (do NOT re-run Step 1).")
print("="*55)
import os; os.kill(os.getpid(), 9)


## Cell 9-alt Step 2 — Convert Gemma 3 1B to TFLite

> Run after Step 1 auto-restart. Takes 10–30 min.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

LITERT_DIR = '/content/litert_output'
os.makedirs(LITERT_DIR, exist_ok=True)

# ── Verify litert-torch imports cleanly ───────────────────────────────────────
import torch
print(f"torch {torch.__version__}")
from litert_torch.generative.utilities import converter as _conv
print("✓ litert_torch.generative.utilities.converter imported")

# ── Locate Gemma 3 weights from HF cache (cached from Cell 4) ─────────────────
HF_TOKEN_FILE = '/content/drive/MyDrive/hf_token.txt'
if os.path.exists(HF_TOKEN_FILE):
    with open(HF_TOKEN_FILE) as _f:
        os.environ['HF_TOKEN'] = _f.read().strip()

from huggingface_hub import snapshot_download
print("Locating Gemma 3 1B IT weights (HF cache, no re-download)...")
KERAS_DIR = snapshot_download(
    repo_id='google/gemma-3-1b-it',
    token=os.environ.get('HF_TOKEN'),
    ignore_patterns=['*.gguf', 'flax_model*', 'tf_model*', 'rust_model*'],
)
print(f"✓ {KERAS_DIR}")

# ── Load Gemma 3 via litert-torch ─────────────────────────────────────────────
# litert-torch 0.8.0 was released before Gemma 3 (March 2025).
# If gemma3 module is missing, remove the ==0.8.0 pin in Step 1 and retry.
try:
    from litert_torch.generative.examples.gemma import gemma3 as _gm
    print("✓ gemma3 module found")
except ImportError as _e:
    print(f"✗ gemma3 module not found in litert-torch 0.8.0: {_e}")
    print("  Fix: edit Step 1, change 'litert-torch==0.8.0' to 'litert-torch'")
    print("  Then: Factory reset → Step 1 → Step 2")
    raise

# Find build function for 1B model
build_fn = None
for _name in ['build_1b_model', 'build_1b_instruct_model', 'build_model']:
    if hasattr(_gm, _name):
        build_fn = getattr(_gm, _name)
        print(f"✓ Using _gm.{_name}")
        break
if build_fn is None:
    available = [x for x in dir(_gm) if not x.startswith('_')]
    print(f"  Available in gemma3 module: {available}")
    raise RuntimeError("No 1B build function found — check output above and report")

SEQ_LEN = 512  # Hard limit for litert-torch; >512 triggers XlaCallModule error
print(f"\nBuilding Gemma 3 1B model (SEQ_LEN={SEQ_LEN})...")
model = build_fn(KERAS_DIR, mask_cache_size=SEQ_LEN)
model.eval()
param_count = sum(p.numel() for p in model.parameters()) / 1e9
print(f"✓ {param_count:.2f}B params")

# ── ExportConfig ──────────────────────────────────────────────────────────────
_EC = _conv.ExportConfig
_export_cfg = None
_ec_mod = getattr(_gm, 'export_cfg', None)
if _ec_mod:
    for _attr in ['get_export_config', 'get_default_export_config', 'export_config']:
        _obj = getattr(_ec_mod, _attr, None)
        if _obj is None:
            continue
        _export_cfg = _obj() if callable(_obj) else _obj
        if _export_cfg is not None:
            print(f"✓ ExportConfig from _gm.export_cfg.{_attr}")
            break
if _export_cfg is None:
    _export_cfg = _EC()
    print("✓ ExportConfig() with all defaults")
print(f"ExportConfig: {_export_cfg}")

# ── Convert to TFLite ─────────────────────────────────────────────────────────
print("\nConverting to DYNAMIC_INT8 TFLite (10–30 min)...")
_conv.convert_to_tflite(
    model,
    LITERT_DIR,
    'cooking_assistant',
    SEQ_LEN,
    SEQ_LEN,
    quantize='dynamic_int8',
    export_config=_export_cfg,
)

# ── Find output file ──────────────────────────────────────────────────────────
_candidates = [
    f"{LITERT_DIR}/{f}" for f in os.listdir(LITERT_DIR)
    if f.endswith('.tflite') or f.endswith('.litert')
]
if not _candidates:
    raise RuntimeError("No .tflite/.litert output found in LITERT_DIR")
TFLITE_PATH = max(_candidates, key=os.path.getmtime)
print(f"✓ {os.path.basename(TFLITE_PATH)} ({os.path.getsize(TFLITE_PATH)/1e6:.0f} MB)")

# ── Validate prefill/decode signatures ───────────────────────────────────────
import tensorflow as tf
interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
sigs = interp.get_signature_list()
print(f"Signatures: {list(sigs.keys())}")
has_decode  = any('decode'  in s for s in sigs)
has_prefill = any('prefill' in s for s in sigs)
print(f"  decode  : {'✓' if has_decode  else '✗ MISSING'}")
print(f"  prefill : {'✓' if has_prefill else '✗ MISSING'}")
if not (has_decode and has_prefill):
    raise RuntimeError("Missing prefill/decode signatures — wrong executor format")

# ── Back up .tflite to Drive ──────────────────────────────────────────────────
import tarfile
BACKUP_TAR = '/content/drive/MyDrive/litert_output_cpu_gemma3.tar'
with tarfile.open(BACKUP_TAR, 'w:') as t:
    t.add(TFLITE_PATH, arcname='cooking_assistant.tflite')
print(f"✓ Backed up to Drive: {os.path.getsize(BACKUP_TAR)/1e6:.0f} MB")
print("\n→ Switch to main session → Cell 9b → 10d → 10b → 11")


---
## Back to main session

Switch back to the original Colab session (or open a new one with the notebook).  
The main session still has MediaPipe and the original packages installed.


## Cell 9b — Restore .tflite from Drive backup

Skip if `LITERT_DIR` is already populated from Step 2.

In [ ]:
import os, tarfile, shutil

BACKUP_TAR = '/content/drive/MyDrive/litert_output_cpu_gemma3.tar'
LITERT_DIR = '/content/litert_output'
HF_DIR     = '/content/hf_merged'  # reused for tokenizer path

assert os.path.exists(BACKUP_TAR), (
    f"Backup not found: {BACKUP_TAR}\n"
    "Run Cell 9-alt Step 1+2 first."
)

if os.path.exists(LITERT_DIR):
    shutil.rmtree(LITERT_DIR)
os.makedirs(LITERT_DIR)

print(f"Restoring from {BACKUP_TAR} ({os.path.getsize(BACKUP_TAR)/1e6:.0f} MB)...")
with tarfile.open(BACKUP_TAR, 'r:') as t:
    t.extractall(LITERT_DIR)

tflite  = [f for f in os.listdir(LITERT_DIR) if f.endswith('.tflite')]
w_files = [f for f in os.listdir(LITERT_DIR) if f.endswith('.w')]
print(f"  .tflite : {tflite}")
print(f"  .w files: {len(w_files)}  (should be 0 for CPU)")
if tflite and len(w_files) == 0:
    print("✓ Restored (CPU format — no .w files)")
else:
    print("✗ Unexpected state — check entries above")
    raise RuntimeError("Bad restore state")

# Tokenizer: get from HF weights (re-downloaded or cached)
os.makedirs(HF_DIR, exist_ok=True)
tok_dest = f"{HF_DIR}/tokenizer.model"
if not os.path.exists(tok_dest):
    # Copy from HF cache
    from huggingface_hub import snapshot_download
    HF_TOKEN_FILE = '/content/drive/MyDrive/hf_token.txt'
    tok = open(HF_TOKEN_FILE).read().strip() if os.path.exists(HF_TOKEN_FILE) else None
    KERAS_DIR = snapshot_download('google/gemma-3-1b-it', token=tok,
                                  ignore_patterns=['*.gguf','flax_model*','tf_model*','rust_model*'])
    # Gemma 3 tokenizer: try tokenizer.model (sentencepiece) or tokenizer.json
    for fname in ['tokenizer.model', 'tokenizer.json']:
        src = f"{KERAS_DIR}/{fname}"
        if os.path.exists(src):
            import shutil as _sh
            _sh.copy(src, tok_dest)
            print(f"✓ Tokenizer: {fname} ({os.path.getsize(tok_dest)/1024:.0f} KB)")
            break
    else:
        print("⚠ No tokenizer.model found — Cell 10d may fail. Check KERAS_DIR contents.")
else:
    print(f"✓ Tokenizer already present ({os.path.getsize(tok_dest)/1024:.0f} KB)")

print("\n✓ Ready → Cell 10d")


## Cell 10d — Bundle `.tflite` + tokenizer → `.task`

In [ ]:
import os, inspect, zipfile
from mediapipe.tasks.python.genai import bundler

LITERT_DIR = '/content/litert_output'
HF_DIR     = '/content/hf_merged'
FINAL_TASK = '/content/cooking_assistant_gemma3.task'
TOKENIZER  = f'{HF_DIR}/tokenizer.model'

# ── Find newest .tflite (litert-torch names it after quant+seqlen) ─────────────
_candidates = [
    f"{LITERT_DIR}/{f}" for f in os.listdir(LITERT_DIR)
    if f.endswith('.tflite') or f.endswith('.litert')
]
assert _candidates, f"No .tflite in {LITERT_DIR} — run Cell 9b first"
TFLITE_PATH = max(_candidates, key=os.path.getmtime)
print(f"Using: {os.path.basename(TFLITE_PATH)}")
print(f".tflite : {os.path.getsize(TFLITE_PATH)/1e6:.0f} MB")

assert os.path.exists(TOKENIZER), (
    f"Tokenizer not found: {TOKENIZER}\n"
    "Re-run Cell 9b — it will copy tokenizer from HF cache."
)
print(f"tokenizer: {TOKENIZER} ({os.path.getsize(TOKENIZER)/1024:.0f} KB)")

# ── BundleConfig — Gemma 3 uses same chat template as Gemma 2 ─────────────────
sig_params = list(inspect.signature(bundler.BundleConfig.__init__).parameters)
print(f"BundleConfig params: {sig_params}")

if 'prompt_prefix_user' in sig_params:
    # 0.10.33+ API
    bundle_kwargs = dict(
        tflite_model        = TFLITE_PATH,
        tokenizer_model     = TOKENIZER,
        start_token         = '<bos>',
        stop_tokens         = ['<eos>', '<end_of_turn>'],
        prompt_prefix_user  = '<start_of_turn>user\n',
        prompt_suffix_user  = '<end_of_turn>\n',
        prompt_prefix_model = '<start_of_turn>model\n',
        prompt_suffix_model = '<end_of_turn>',
    )
    print("Using 0.10.33+ API (per-role)")
else:
    # 0.10.21 legacy API
    bundle_kwargs = dict(
        tflite_model  = TFLITE_PATH,
        tokenizer_model = TOKENIZER,
        start_token   = '<bos>',
        stop_tokens   = ['<eos>', '<end_of_turn>'],
        prompt_prefix = '<start_of_turn>user\n',
        prompt_suffix = '<end_of_turn>\n<start_of_turn>model\n',
    )
    print("Using 0.10.21 legacy API")

print("Creating .task...")
bundler.create_bundle(bundler.BundleConfig(
    **bundle_kwargs,
    output_filename=FINAL_TASK,
))

# ── Validate .task contents ────────────────────────────────────────────────────
task_mb = os.path.getsize(FINAL_TASK) / 1e6
print(f"\n✓ .task created: {FINAL_TASK} ({task_mb:.0f} MB)")

with zipfile.ZipFile(FINAL_TASK, 'r') as z:
    entries = z.namelist()
    print("Contents:")
    for e in entries:
        print(f"  {e:40s}  {z.getinfo(e).file_size/1e6:.1f} MB")
    has_tflite    = any('PREFILL_DECODE' in e or e.endswith('.tflite') for e in entries)
    has_tokenizer = any('TOKENIZER' in e.upper() or e.endswith('.model') or e.endswith('.json') for e in entries)
    has_w         = any(e.endswith('.w') for e in entries)
    print()
    print(f"  TF_LITE_PREFILL_DECODE : {'✓' if has_tflite    else '✗ MISSING'}")
    print(f"  TOKENIZER              : {'✓' if has_tokenizer else '✗ MISSING'}")
    print(f"  .w files               : {'⚠ present (unexpected)' if has_w else '✓ none (correct)'}")

if has_tflite and has_tokenizer and not has_w:
    print("\n✓ .task structure correct — ready for Cell 10b → 11")
else:
    print("\n✗ Unexpected structure — check entries above")


## Cell 10b — Validate `.task`

In [ ]:
import os, zipfile

FINAL_TASK = '/content/cooking_assistant_gemma3.task'
assert os.path.exists(FINAL_TASK), f"No .task at {FINAL_TASK} — run Cell 10d first"

with zipfile.ZipFile(FINAL_TASK, 'r') as z:
    names = z.namelist()
    total_mb = sum(z.getinfo(n).file_size for n in names) / 1e6
    print(f"Archive: {len(names)} entries, {total_mb:.0f} MB uncompressed")
    for n in names:
        print(f"  {n}")

print(f"\nFile size: {os.path.getsize(FINAL_TASK)/1e6:.0f} MB")
print("\n✓ Validation complete — ready for Cell 11")


## Cell 11 — Save to Drive + download

In [ ]:
import os, shutil
from google.colab import files

FINAL_TASK = '/content/cooking_assistant_gemma3.task'
DRIVE_DEST = '/content/drive/MyDrive/cooking_assistant_gemma3.task'

# Save to Drive
shutil.copy(FINAL_TASK, DRIVE_DEST)
print(f"✓ Saved to Drive: {DRIVE_DEST} ({os.path.getsize(DRIVE_DEST)/1e6:.0f} MB)")

# Download directly
print("Downloading to local machine...")
files.download(FINAL_TASK)
print("\nDone!")
print("\n--- iOS integration ---")
print("1. Drag cooking_assistant_gemma3.task into Xcode project")
print("2. AppConfig.modelFileName is already set to 'cooking_assistant_gemma3'")
print("3. Delete app from iPhone and reinstall to clear Documents/ cache")


---
## Notes

- **Fast path model:** `litert-community/Gemma3-1B-IT` — `Gemma3-1B-IT_multi-prefill-seq_q8_ekv1280.task` (~1 GB, dynamic_int8, context 1280)
- **Full conversion model:** `google/gemma-3-1b-it` → litert-torch → DYNAMIC_INT8 @ SEQ_LEN=512
- **Quantization:** DYNAMIC_INT8 required for iOS — INT4 causes garbled output (confirmed on Gemma 2 2B, same issue expected)
- **Chat template:** `<start_of_turn>user\n…<end_of_turn>\n<start_of_turn>model\n` — the `\n` after `user` and `model` is critical
- **iOS model file:** `cooking_assistant_gemma3.task` → matches `AppConfig.modelFileName`
- **On iOS, delete and reinstall the app** after replacing the `.task` — the Documents/ cache won't update otherwise
- **litert-torch 0.8.0 gemma3 module:** Confirmed supported (Gemma 3 support added March 2025)
- **SEQ_LEN:** 512 hard limit for full conversion path; fast path model uses ekv1280 (larger context, fully compatible with MediaPipe)